# Phase 5: matched development evaluation

**Research question:** Can a learned return ranker improve stock ordering, and can a learned risk classifier improve Low/Medium/High volatility prediction on later chronological blocks?

Four outer development folds use the frozen Phase 4 protocol. Return is a same-date peer class; risk classes use cutoffs computed from each fold's training outcomes. All signals use information available through session *t* and are available after that close. The 2024–2025 reused test period is excluded from selection.

From the repository root, use **Restart & Run All** in Jupyter with a Python kernel that has the project requirements installed (`ipykernel` is needed if your environment has no notebook kernel). Code cells use only ordinary Python. The source package verifies the frozen inputs, fits any missing Phase 5 model artifacts, reviews matched predictions, and writes tables and SVG figures under `reports/generated/current/model_evaluation_package/`. Existing verified fits are reused. On a fresh checkout, generate the Phase 3 dataset and Phase 4 protocol with the commands in `decision.md` before opening this notebook. The model fits can take several minutes.

In [3]:
from pathlib import Path
import os
import sys
import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file() and (p / 'src' / 'modeling').is_dir()), None)
if root is None:
    raise RuntimeError('Open the notebook within the HalalifyAI repository.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.model_training.build_model_evaluation_report import build_results_package, show_table, show_svg
summary, tables, artifact_dir = build_results_package()
print('Research rows:', summary['data']['rows'], 'companies:', summary['data']['companies'])
print('Return eligible:', summary['data']['return_eligible'], 'Risk eligible:', summary['data']['risk_eligible'])
print('Figures and tables:', artifact_dir)

RuntimeError: Open the notebook within the HalalifyAI repository.

## Cohort and fold boundaries

Each fold uses shared market session dates. Training outcomes end before evaluation starts; the risk cutoffs shown here are computed from outer training outcomes only. Model input columns come from the frozen feature groups and exclude labels, target dates, identifiers, and provenance.

In [ ]:
folds = tables['folds.csv'].copy()
show_table(folds.round({'risk_low_cutoff': 4, 'risk_high_cutoff': 4}))

## Return: same-date stock ordering

The learned score is `P(Higher) - P(Lower)`; it is an ordering score, **not** a percentage return. The training-class prior is constant across stocks on a date, so its rank correlation and top-10 selection are undefined. Trailing 20-day momentum is a simple rule-based benchmark, not a fitted model. Rank correlation is the primary return metric. Top-10 forward-close spreads are research selection diagnostics.

In [ ]:
return_metrics = tables['return_fold_metrics.csv'].copy()
show_table(return_metrics.round({'mean_daily_spearman': 4, 'mean_top10_minus_universe': 4, 'macro_f1': 4}))
show_svg(artifact_dir / 'return_fold_spearman.svg')
print('Equal-weight mean fold Spearman:', {name: round(value, 4) for name, value in summary['return_mean_fold_spearman'].items()})

**Return decision for this development review:** No return approach is selected. Momentum's mean rank correlation is effectively zero. Logistic Regression's higher mean varies across folds, and the boosted tree has almost no mean ordering signal. A new or revised approach requires a separately frozen evaluation before a final generalization claim.

## Risk: class quality and severe misses

Macro F1 is the primary metric. High recall and **High predicted Low** reveal risk underestimation that the average can hide. Class prior and trailing volatility use the same outer rows and fold-specific risk boundaries as the learned models. Log loss and multiclass Brier apply only where probabilities exist.

In [ ]:
risk_metrics = tables['risk_fold_metrics.csv'].copy()
show_table(risk_metrics.round({'macro_f1': 4, 'high_recall': 4, 'ordinal_mae': 4, 'log_loss': 4, 'multiclass_brier': 4}))
show_svg(artifact_dir / 'risk_fold_f1.svg')
risk_summary = pd.DataFrame(summary['risk_summary']).T
show_table(risk_summary[['mean_fold_macro_f1', 'mean_fold_high_recall', 'total_high_predicted_low', 'pooled_high_predicted_low_rate', 'mean_fold_ordinal_mae']].round(4).reset_index(names='method'))

### Per-class errors and confusion matrices

Rows of each confusion matrix are observed risk classes; columns are predicted classes. All four folds appear below so failure cases remain visible beside average scores.

In [ ]:
risk_classes = tables['risk_class_metrics.csv']
show_table(risk_classes[risk_classes.method.isin(['LogisticRegression', 'HistGradientBoosting'])].round(4))
confusion = tables['risk_confusion_matrices.csv']
for fold_number in range(1, 5):
    for method in ('LogisticRegression', 'HistGradientBoosting'):
        subset = confusion[(confusion.fold == fold_number) & (confusion.method == method)]
        matrix = subset.pivot(index='observed', columns='predicted', values='count').reindex(index=['Low', 'Medium', 'High'], columns=['Low', 'Medium', 'High'])
        print(f'Fold {fold_number}: {method}; observed rows, predicted columns')
        show_table(matrix.reset_index())

**Risk interpretation:** The boosted tree is a provisional candidate. Its mean macro F1 lead over Logistic Regression is small and changes direction across folds. It has fewer High-to-Low misses than Logistic Regression; trailing volatility has slightly higher High recall and slightly fewer such misses than the tree. Step 5.11 must weigh these outcomes before a final Phase 5 conclusion. Numerical MAE/RMSE/R² in the risk review apply only to the direct trailing-volatility forecast; classifier probabilities are not numerical volatility forecasts.

## Reproducibility and limits

The summary lists exact dataset, protocol, model-prediction and artifact hashes plus library versions. The source code rejects mismatched saved fits. Generated files are intentionally Git-ignored; source modules, this notebook, and `decision.md` are reviewable in Git. This is an academic research comparison, not a compliant portfolio or executable backtest.

In [ ]:
print('Dataset SHA-256:', summary['dataset_sha256'])
print('Protocol SHA-256:', summary['protocol_sha256'])
print('Environment:', summary['environment'])
print('Prediction hashes:', summary['model_prediction_sha256'])
print('Artifact hashes:', summary['artifacts'])
print('Limitations:')
for item in summary['limitations']:
    print('•', item)